# Большой демо-ноутбук Yandex AI Studio для экономических данных

Этот ноутбук показывает один связный сценарий: как использовать OpenAI-совместимый Responses API в Yandex AI Studio для задач, похожих на работу экономиста-аналитика. Мы будем задавать модели вопросы про ВВП, подключать встроенный веб-поиск, получать структурированный ответ, строить графики, вызывать внешние функции и переносить функцию в MCP-сервер.

Код специально написан просто. В реальном проекте многие блоки стоило бы вынести в модули, добавить обработку ошибок, логирование и тесты. Здесь цель другая: чтобы студент мог выполнить ячейку, увидеть результат и понять, какой минимальный фрагмент отвечает за конкретную возможность платформы.

## 0. Установка библиотек

В первой ячейке устанавливаем библиотеки, которые понадобятся в разных частях демонстрации. После установки в Jupyter часто полезно перезапустить Kernel, особенно если пакет openai-agents или openai был обновлён прямо во время занятия.

Обратите внимание: ноутбук не запускает все дорогие или долгие операции автоматически. Студенты запускают ячейки по очереди, когда преподаватель объясняет соответствующий раздел.

In [ ]:
%pip install --upgrade openai openai-agents python-dotenv pydantic pandas matplotlib requests pyvis networkx

## 1. Авторизация и базовая настройка клиента

Yandex AI Studio предоставляет OpenAI-совместимый API. Поэтому мы используем обычный класс OpenAI, но указываем base_url Yandex Cloud и передаём project=folder_id.

В файле .env должны быть переменные folder_id, api_key и openweathermap_appid. Мы вызываем load_dotenv(override=True), чтобы значения из .env переопределяли переменные окружения текущего процесса. Для учебной демонстрации это удобно: поменяли .env, перезапустили ячейку, и ноутбук взял новые значения.

In [ ]:
import os
import json
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, Image, display
from openai import OpenAI

load_dotenv(override=True)

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]

model_qwen36 = f"gpt://{folder_id}/qwen3.6-35b-a3b/latest"
model_qwen3 = f"gpt://{folder_id}/qwen3-235b-a22b-fp8/latest"
model_deepseek = f"gpt://{folder_id}/deepseek-v32/latest"

client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id,
)

def printx(text):
    display(Markdown(text))

print(f"Клиент создан. folder_id начинается с: {folder_id[:8]}...")
print("Qwen 3.6:", model_qwen36)
print("Qwen 3:", model_qwen3)
print("DeepSeek:", model_deepseek)

## 2. Первый запрос через Responses API

Начнём с самого простого вызова модели. Мы задаём экономический вопрос на естественном языке и получаем текстовый ответ.

Важная мысль: модель отвечает на основе своих знаний и доступного контекста. Если нужны свежие числа, ссылки на источники или проверка фактов, дальше мы подключим веб-поиск. Но для знакомства с API полезно сначала увидеть минимальный запрос.

In [ ]:
gdp_question = "Покажи динамику ВВП России за 2010-2025 гг. Дай краткий комментарий к основным изменениям."

response = client.responses.create(
    model=model_qwen36,
    input=gdp_question,
)

printx(response.output_text)

### Сравним несколько моделей

Ниже простой цикл по трём моделям. Это не строгий бенчмарк, а учебная демонстрация: одинаковый вопрос можно отправлять разным моделям и сравнивать стиль ответа, полноту, аккуратность с числами и объяснениями.

На практике для точного сравнения нужно фиксировать параметры генерации, несколько раз повторять запрос и проверять ответы по источникам.

In [ ]:
models = {
    "Qwen 3.6": model_qwen36,
    "Qwen 3": model_qwen3,
    "DeepSeek": model_deepseek,
}

for name, model_uri in models.items():
    print(f"\n===== {name} =====")
    res = client.responses.create(
        model=model_uri,
        input="Кратко объясни, какие факторы влияли на динамику ВВП России в 2010-2025 гг.",
    )
    print(res.output_text[:1500])

## 3. Встроенный инструмент Web Search

Теперь подключим внутренний инструмент веб-поиска. Модель сама формулирует поисковый запрос, получает результаты и использует их при ответе.

Здесь мы делаем тот же запрос про ВВП России. После ответа посмотрим, какой поисковый запрос был выполнен, и какие ссылки попали в аннотации ответа. Это важно для аналитических задач: студент должен видеть не только красивый текст, но и источник фактов.

In [ ]:
web_search_tool = {"type": "web_search", "search_context_size": "medium"}

res_ws = client.responses.create(
    model=model_qwen3,
    instructions=(
        "Ты экономический аналитик. Когда используешь данные из интернета, "
        "отвечай аккуратно и упоминай, что именно найдено в источниках."
    ),
    tools=[web_search_tool],
    input=gdp_question,
)

printx(res_ws.output_text)

### Что именно искала модель

Ответ Responses API состоит из нескольких элементов. Среди них может быть web_search_call — запись о вызове поискового инструмента. Ниже мы просто пробегаем по элементам ответа и печатаем поисковые запросы, если они есть.

In [ ]:
for item in res_ws.output:
    if item.type == "web_search_call":
        print("Поисковый запрос модели:", item.action.query)

### Ссылки из ответа

В текстовом сообщении могут быть аннотации со ссылками. Их удобно извлекать отдельно: например, чтобы показать студентам, на какие источники опирался ответ, или чтобы позже сохранить список источников в отчёт.

In [ ]:
for item in res_ws.output:
    if item.type == "message":
        for content in item.content:
            for ann in getattr(content, "annotations", []) or []:
                title = getattr(ann, "title", "без названия")
                url = getattr(ann, "url", "")
                if url:
                    print(f"{title} - {url}")

## 4. Структурированный ответ и локальный график

Иногда нам нужен не текст, а данные в предсказуемой форме. Например, список годов и значений ВВП, который можно сразу превратить в таблицу и построить график.

Для этого используем Pydantic-модель как схему ответа. Модель должна вернуть объект, соответствующий этой схеме. Здесь мы просим значения в трлн рублей. Это учебный пример: для серьёзной работы обязательно нужно уточнять методику, источник, текущие или постоянные цены и дату обновления.

In [ ]:
from pydantic import BaseModel, Field
import pandas as pd
import matplotlib.pyplot as plt

class GDPPoint(BaseModel):
    year: int = Field(description="Год")
    gdp_trillion_rub: float = Field(description="ВВП России, трлн рублей")

class GDPSeries(BaseModel):
    country: str
    unit: str
    points: list[GDPPoint]
    comment: str

structured = client.responses.parse(
    model=model_qwen3,
    instructions=(
        "Верни только структурированные данные. "
        "Если точные данные за 2024-2025 недоступны, используй разумную оценку и поясни это в comment."
    ),
    input="Дай динамику номинального ВВП России за 2010-2025 гг. в трлн рублей.",
    text_format=GDPSeries,
)

gdp_data = structured.output_parsed
print(gdp_data.comment)

df_gdp = pd.DataFrame([p.model_dump() for p in gdp_data.points]).sort_values("year")
df_gdp.head()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_gdp["year"], df_gdp["gdp_trillion_rub"], marker="o")
plt.title("ВВП России, 2010-2025")
plt.xlabel("Год")
plt.ylabel(gdp_data.unit)
plt.grid(True, alpha=0.3)
plt.show()

## 5. Tool Calling: локальная функция погоды

Tool Calling означает, что модель может не только писать текст, но и попросить наше приложение вызвать функцию. В этом разделе функция простая: она обращается к OpenWeatherMap и возвращает текущую погоду по городу.

Мы используем небольшой класс Agent из файла Agent.py, чтобы не писать весь цикл обработки function calls в ноутбуке. Сам инструмент описан как Pydantic-класс: поля класса становятся JSON Schema для модели, а метод process выполняет реальный код.

Демо-задание: получить погоду в 10 крупнейших городах России и сделать краткую таблицу.

In [ ]:
import requests
from pydantic import BaseModel, Field

from Agent import Agent, create_client

openweathermap_appid = os.environ["openweathermap_appid"]
agent_client = create_client()

class GetWeather(BaseModel):
    """Получить текущую погоду в городе через OpenWeatherMap."""

    city: str = Field(description="Название города на английском, например Moscow")

    def process(self, session_id: str) -> str:
        url = "https://api.openweathermap.org/data/2.5/weather"
        params = {
            "q": self.city,
            "appid": openweathermap_appid,
            "units": "metric",
            "lang": "ru",
        }
        data = requests.get(url, params=params, timeout=15).json()
        result = {
            "city": data.get("name", self.city),
            "temperature_c": data["main"]["temp"],
            "feels_like_c": data["main"]["feels_like"],
            "humidity_percent": data["main"]["humidity"],
            "description": data["weather"][0]["description"],
            "wind_m_s": data["wind"]["speed"],
        }
        return json.dumps(result, ensure_ascii=False)

weather_agent = Agent(
    client=agent_client,
    model=model_qwen3,
    instruction=(
        "Ты ассистент для учебной демонстрации tool calling. "
        "Для каждого города вызывай инструмент GetWeather. "
        "В финальном ответе верни компактную Markdown-таблицу на русском языке."
    ),
    tools=[GetWeather],
    verbose=True,
)

weather_response = weather_agent(
    "Получи текущую погоду в 10 крупнейших городах России: Moscow, Saint Petersburg, "
    "Novosibirsk, Yekaterinburg, Kazan, Nizhny Novgorod, Chelyabinsk, Krasnoyarsk, Samara, Ufa."
)

printx(weather_response.output_text)

## 6. Перенос погодной функции в MCP-сервер

Локальная функция удобна для ноутбука, но в реальном приложении инструмент часто живёт отдельно: в облачной функции, контейнере или внутреннем сервисе. Тогда модель подключается к нему через MCP.

В папке weather_func лежит минимальный пример: index.py принимает city и вызывает OpenWeatherMap, requirements.txt содержит requests, deploy.ps1 упаковывает функцию и создаёт MCP Gateway.

Скрипт после создания MCP Gateway печатает его ID. Когда Gateway станет ACTIVE, нужно получить baseDomain и добавить /sse. Полученный URL можно положить в .env как weather_mcp_url или вставить в ячейку ниже.

In [ ]:
# Пример запуска в терминале PowerShell из корня проекта:
# .\weather_func\deploy.ps1 -ServiceAccountId "<service-account-id>"

WEATHER_MCP_URL = os.getenv("weather_mcp_url", "PASTE_YOUR_MCP_SSE_URL_HERE")

if WEATHER_MCP_URL == "PASTE_YOUR_MCP_SSE_URL_HERE":
    print("Сначала задеплойте weather_func и вставьте MCP URL в WEATHER_MCP_URL или .env.")
else:
    mcp_weather_tool = {
        "type": "mcp",
        "server_label": "weather",
        "server_url": WEATHER_MCP_URL,
        "require_approval": "never",
    }

    mcp_response = client.responses.create(
        model=model_qwen3,
        tools=[mcp_weather_tool],
        input=(
            "Через MCP-инструмент получи текущую погоду для Moscow, Kazan и Novosibirsk. "
            "Верни короткую таблицу на русском."
        ),
    )

    printx(mcp_response.output_text)

## 7. Code Interpreter: анализ CSV-файла с ВВП стран

Теперь покажем встроенный Code Interpreter. Мы загрузим файл sample_data/gdp_by_country.csv в контейнер и попросим модель написать Python-код, выбрать 10 стран с самым большим средним ВВП и построить график динамики.

Отличие от локального matplotlib-раздела: здесь код выполняет не наш ноутбук, а среда Code Interpreter. Это удобно, когда мы хотим дать модели файл и попросить её самостоятельно провести небольшой анализ.

In [ ]:
def inspect_code_interpreter_response(response, download_dir="out"):
    download_dir = Path(download_dir)
    download_dir.mkdir(exist_ok=True)
    downloaded_files = []

    print("ID ответа:", response.id)
    print("Элементы ответа:", [item.type for item in response.output])

    for item in response.output:
        if item.type == "code_interpreter_call":
            print("\nКод, который выполнила модель:\n")
            print(item.code)

        if item.type == "message":
            for content in item.content:
                for ann in getattr(content, "annotations", []) or []:
                    if getattr(ann, "type", None) == "container_file_citation":
                        local_path = download_dir / ann.filename
                        client.files.content(ann.file_id).write_to_file(local_path)
                        downloaded_files.append(local_path)
                        print("Скачан файл:", local_path)

    return downloaded_files

In [ ]:
with open("sample_data/gdp_by_country.csv", "rb") as file_handle:
    gdp_file = client.files.create(file=file_handle, purpose="assistants")

container = client.containers.create(
    name="gdp-country-demo",
    expires_after={"anchor": "last_active_at", "minutes": 20},
    file_ids=[gdp_file.id],
)

print("CSV загружен:", gdp_file.id)
print("Container ID:", container.id)

In [ ]:
ci_response = client.responses.create(
    model=model_qwen3,
    instructions=(
        "Ты Data Scientist. Работай с файлом gdp_by_country.csv через pandas. "
        "Не придумывай данные, используй только загруженный CSV."
    ),
    input=(
        "Построй график динамики ВВП для топ-10 стран с самым большим средним ВВП "
        "по всем доступным годам в таблице. Сохрани график в PNG-файл и кратко объясни результат."
    ),
    include=["code_interpreter_call.outputs"],
    tools=[{"type": "code_interpreter", "container": container.id}],
)

printx(ci_response.output_text)

In [ ]:
downloaded = inspect_code_interpreter_response(ci_response)

for file_path in downloaded:
    if file_path.suffix.lower() in {".png", ".jpg", ".jpeg"}:
        display(Image(filename=str(file_path)))

## 8. OpenAI Agents SDK: Deep Research, один агент

В этой части используем OpenAI Agents SDK поверх Yandex AI Studio. Идея: вместо одного прямого запроса мы создаём агента с инструкциями и инструментами. Агент может искать информацию, сохранять заметки, строить граф понятий и использовать Code Interpreter для отчёта.

Этот пример ближе к настоящему агентному приложению и может выполняться дольше обычных запросов. Для занятия можно уменьшить тему или ограничить число шагов, если нужно показать механику быстрее.

In [ ]:
from openai import AsyncOpenAI
from agents import Agent as SDKAgent
from agents import Runner, WebSearchTool, CodeInterpreterTool, function_tool, set_tracing_disabled
from agents.models.openai_responses import OpenAIResponsesModel

set_tracing_disabled(True)

async_client = AsyncOpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id,
)

yandex_model = OpenAIResponsesModel(
    model=model_qwen3,
    openai_client=async_client,
)

print("Agents SDK model is ready")

In [ ]:
research_notes = []
concept_edges = []

@function_tool
def save_note(topic: str, content: str, source: str = "") -> str:
    """Save a research note for later use."""
    research_notes.append({"topic": topic, "content": content, "source": source})
    return f"Saved note {len(research_notes)}: {topic}"

@function_tool
def save_concept_edge(source: str, relation: str, target: str) -> str:
    """Save one relationship for the concept graph."""
    concept_edges.append({"source": source, "relation": relation, "target": target})
    return f"Saved edge: {source} - {relation} - {target}"

@function_tool
def get_research_memory() -> str:
    """Return saved notes and concept edges."""
    return json.dumps({"notes": research_notes, "concept_edges": concept_edges}, ensure_ascii=False, indent=2)

def reset_research_memory():
    research_notes.clear()
    concept_edges.clear()

print("Research memory tools are ready")

In [ ]:
from agents import RunHooks, RunContextWrapper, Tool

class VerboseHooks(RunHooks):
    async def on_agent_start(self, context: RunContextWrapper, agent):
        print(f"Агент запущен: {agent.name}")

    async def on_agent_end(self, context: RunContextWrapper, agent, output):
        print(f"Агент завершён: {agent.name}")

    async def on_tool_start(self, context: RunContextWrapper, agent, tool: Tool):
        print(f"Вызов инструмента: {tool.name}")

    async def on_tool_end(self, context: RunContextWrapper, agent, tool: Tool, result):
        print(f"Инструмент завершён: {tool.name}")

verbose_hooks = VerboseHooks()

In [ ]:
deep_research_agent = SDKAgent(
    name="DeepResearchAgent",
    model=yandex_model,
    instructions="""
Ты агент глубокого исследования для учебного демо.
Исследуй тему, используй веб-поиск, сохраняй заметки и связи понятий.
Перед финальным ответом вызови get_research_memory.
В финале дай структурированный обзор на русском языке.
""",
    tools=[
        WebSearchTool(),
        save_note,
        save_concept_edge,
        get_research_memory,
        CodeInterpreterTool(tool_config={"type": "code_interpreter", "container": {"type": "auto"}}),
    ],
)

async def deep_research(topic):
    reset_research_memory()
    return await Runner.run(
        deep_research_agent,
        f"Проведи исследование по теме: {topic}",
        hooks=verbose_hooks,
        max_turns=40,
    )

single_agent_result = await deep_research("как ИИ-агенты помогают экономистам анализировать макроэкономические данные")
printx(single_agent_result.final_output)

### Скачивание файлов из Code Interpreter агента

Если агент создал файлы через Code Interpreter, их можно скачать из контейнера. Это отдельный технический шаг: финальный ответ модели может ссылаться на файл, но для работы в локальной папке ноутбука файл нужно явно сохранить.

In [ ]:
def download_agent_files(result, download_dir="out"):
    download_dir = Path(download_dir)
    download_dir.mkdir(exist_ok=True)
    seen = set()

    code_calls = [item.raw_item for item in result.new_items if "ResponseCodeInterpreterToolCall" in str(type(item.raw_item))]
    for call in code_calls:
        for file_info in client.containers.files.list(call.container_id):
            if file_info.id not in seen:
                seen.add(file_info.id)
                local_path = download_dir / Path(file_info.path).name
                client.files.content(file_info.id).write_to_file(local_path)
                print("Скачан файл:", local_path)

download_agent_files(single_agent_result)

### Визуализация графа понятий

Во время исследования агент сохранял связи понятий в список concept_edges. Ниже строим простой HTML-граф через pyvis. Это не обязательная часть Agents SDK, а пример того, как данные, собранные агентом через инструменты, можно использовать обычным Python-кодом.

In [ ]:
from pyvis.network import Network

Path("out").mkdir(exist_ok=True)
net = Network(height="700px", width="100%")

for edge in concept_edges:
    net.add_node(edge["source"])
    net.add_node(edge["target"])
    net.add_edge(edge["source"], edge["target"], title=edge["relation"])

net.show("out/concept_graph.html", notebook=False)
print("Граф сохранён в out/concept_graph.html")

## 9. OpenAI Agents SDK: мультиагентный вариант

Теперь разделим работу между несколькими агентами. Это полезно как учебная модель: один агент планирует, другой ищет источники, третий извлекает связи понятий, четвёртый пишет отчёт.

В реальном проекте мультиагентность не нужна сама по себе. Её стоит использовать, когда разделение ролей делает систему понятнее или надёжнее.

In [ ]:
planner_agent = SDKAgent(
    name="PlannerAgent",
    model=yandex_model,
    instructions="Создай короткий план исследования из 3-5 вопросов. Пиши по-русски.",
)

researcher_agent = SDKAgent(
    name="ResearcherAgent",
    model=yandex_model,
    instructions="Ищи источники по вопросам плана и сохраняй важные факты через save_note.",
    tools=[WebSearchTool(), save_note],
)

graph_agent = SDKAgent(
    name="GraphAnalystAgent",
    model=yandex_model,
    instructions="Прочитай сохранённые заметки и сохрани 5-10 важных связей понятий через save_concept_edge.",
    tools=[get_research_memory, save_concept_edge],
)

writer_agent = SDKAgent(
    name="ReportWriterAgent",
    model=yandex_model,
    instructions="Используй заметки и связи понятий, чтобы написать связный учебный отчёт на русском языке.",
    tools=[get_research_memory],
)

coordinator_agent = SDKAgent(
    name="ResearchCoordinator",
    model=yandex_model,
    instructions="""
Координируй команду исследования.
Вызови инструменты в таком порядке: planner, researcher, graph_analyst, report_writer.
Верни финальный текст report_writer.
""",
    tools=[
        planner_agent.as_tool("planner", "Create a research plan"),
        researcher_agent.as_tool("researcher", "Search and save research notes"),
        graph_agent.as_tool("graph_analyst", "Extract concept graph edges"),
        writer_agent.as_tool("report_writer", "Write the final report"),
    ],
)

print("Multi-agent research team is ready")

In [ ]:
reset_research_memory()

multi_agent_result = await Runner.run(
    coordinator_agent,
    "Исследуй, как ИИ-агенты могут помогать экономистам готовить данные для макроэкономического анализа.",
    max_turns=25,
)

printx(multi_agent_result.final_output)

## 10. Что мы собрали

В одном ноутбуке мы прошли путь от простого запроса к модели до агентного исследования:

1. Создали OpenAI-совместимый клиент Yandex AI Studio.
2. Задали экономический вопрос через Responses API.
3. Подключили Web Search и посмотрели источники.
4. Получили структурированный ответ и построили локальный график.
5. Дали модели локальный инструмент для запроса погоды.
6. Показали, как вынести инструмент в Cloud Function и MCP Gateway.
7. Передали CSV в Code Interpreter и попросили построить график.
8. Запустили Deep Research агента в single-agent и multi-agent вариантах.

Главная практическая идея: LLM становится гораздо полезнее, когда вокруг неё есть инструменты, данные и понятный контур исполнения. Responses API даёт единый способ подключать эти возможности, а Agents SDK помогает собирать более длинные сценарии.